# Appliance Energy Forecasting — Complete Analysis

Short-term forecasting of household appliance electricity demand using the UCI
*Appliances Energy Prediction* dataset.

**Run this notebook top to bottom.** Every section depends on the ones above it,
and the pipeline in Section 2 writes the files that later sections read.

Contents:

| Section | Content |
|---|---|
| 1 | Setup and data |
| 2 | Run the full pipeline |
| 3 | Exploratory analysis and stationarity |
| 4 | Benchmark models |
| 5 | SARIMAX |
| 6 | Feature-based model |
| 7 | Foundation model (Chronos) |
| 8 | Model comparison and significance testing |
| 9 | Save results |

Runtime: roughly 10 minutes on CPU, 6 with a GPU.


## 1. Setup

### 1.1 Get the code

Use **one** of the two cells below.

In [10]:
# Option A - clone from GitHub (recommended)
!git clone https://github.com/muag21/appliance-energy-forecasting.git /content/project
%cd /content/project

fatal: destination path '/content/project' already exists and is not an empty directory.
/content/project


In [11]:
# Option B - upload the project zip instead
# from google.colab import files
# up = files.upload()
# !unzip -q -o {list(up)[0]} -d /content
# %cd /content/project

### 1.2 Install dependencies

Colab already ships numpy, pandas, scikit-learn, statsmodels and torch. Only
Chronos is missing. Do not upgrade the others — it forces a runtime restart.

In [12]:
!pip install -q chronos-forecasting
!pip install -q pytest tabulate

### 1.3 Bootstrap paths

Colab's working directory does not reliably survive between cells. This locates
the project and fixes both `cwd` and `sys.path`. **Re-run after any runtime
restart.**

In [13]:
import sys, os, glob

hits = glob.glob("/content/**/appliance_energy/config.py", recursive=True)
if not hits:
    raise SystemExit("Project not found. Re-run the clone or upload cell above.")

SRC  = os.path.dirname(os.path.dirname(hits[0]))
ROOT = os.path.dirname(SRC)
os.chdir(ROOT)
if SRC not in sys.path:
    sys.path.insert(0, SRC)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from appliance_energy import config, data, evaluation, features, plotting, stationarity
from appliance_energy.models import benchmarks, feature_models, foundation, sarimax

pd.set_option("display.width", 140)
print("project root :", ROOT)
print("import OK")

project root : /content/project
import OK


### 1.4 Verify the test suite

Run this before anything else. The tests enforce the leakage guarantees; if they
fail, no result below is trustworthy.

In [14]:
!python -m pytest -q

.....................................                                    [100%]


## 2. Run the full pipeline

Downloads and caches the data, fits every model, and writes forecasts, metrics
and figures to `outputs/`. Everything after this section reads those files.

`--device auto` uses a GPU if one is attached.

In [15]:
!python scripts/run_pipeline.py --device auto


1. Data
Hourly observations : 3290
Train : 2016-01-11 17:00:00 to 2016-05-13 18:00:00  (2954)
Test  : 2016-05-13 19:00:00 to 2016-05-27 18:00:00  (336)
Target aggregation  : mean

2. Stationarity diagnostics (training sample only)
                series test                 null  statistic      p_value  crit_5pct  lags_used  reject_null_5pct
                 level  ADF    unit root present  -8.760992 2.674521e-14  -2.862529         28              True
                 level KPSS series is stationary   0.061399 1.000000e-01   0.463000         23             False
      first_difference  ADF    unit root present -15.948273 7.406750e-29  -2.862529         28              True
      first_difference KPSS series is stationary   0.041655 1.000000e-01   0.463000         68             False
seasonal_difference_24  ADF    unit root present -12.672453 1.236077e-23  -2.862536         25              True
seasonal_difference_24 KPSS series is stationary   0.015012 1.000000e-01   0.463000       

## 3. Exploratory analysis and stationarity

In [16]:
%cd /content/project
!git pull

/content/project
Already up to date.


In [17]:
frame = data.load_hourly()
y = frame[config.TARGET]
y_train, y_test = data.train_test_split(y)
test_index = y_test.index

print(f"hourly observations : {len(y)}")
print(f"train : {y_train.index.min()} to {y_train.index.max()}  ({len(y_train)})")
print(f"test  : {test_index.min()} to {test_index.max()}  ({len(y_test)})")

hourly observations : 3290
train : 2016-01-11 17:00:00 to 2016-05-13 18:00:00  (2954)
test  : 2016-05-13 19:00:00 to 2016-05-27 18:00:00  (336)


### 3.1 Distribution

The target is strongly right-skewed, which is why MAE and RMSE can rank models
differently later on.

In [18]:
y.describe().round(2).to_frame("value").T

,count,mean,std,min,25%,50%,75%,max
value,3290.0,97.78,81.21,28.33,50.0,63.33,110.0,608.33


In [19]:
print(f"skewness        {y.skew():.2f}")
print(f"excess kurtosis {y.kurtosis():.2f}")
print(f"p95 / median    {y.quantile(0.95) / y.median():.2f}")

skewness        2.39
excess kurtosis 6.33
p95 / median    4.37


### 3.2 Seasonal structure

In [20]:
fig = plotting.plot_series_overview(y)

In [21]:
profile = y.groupby(y.index.hour).mean()
print(f"trough hour {profile.idxmin()} = {profile.min():.1f}")
print(f"peak   hour {profile.idxmax()} = {profile.max():.1f}")
print(f"peak / trough {profile.max() / profile.min():.2f}")

names = ["Mon","Tue","Wed","Thu","Fri","Sat","Sun"]
by_dow = y.groupby(y.index.dayofweek).mean()
print("\nby day:", "  ".join(f"{names[d]} {v:.1f}" for d, v in by_dow.items()))

trough hour 3 = 48.2
peak   hour 18 = 191.8
peak / trough 3.98

by day: Mon 111.5  Tue 87.1  Wed 89.9  Thu 90.4  Fri 105.2  Sat 106.2  Sun 94.9


**Seasonal strength** on the Wang, Smith and Hyndman (2006) scale. Values near
0.3 mean seasonality explains a minority of the variance — the rest is occupant
behaviour, which is the central constraint on every model here.

In [22]:
for period, label in [(config.DAILY_PERIOD, "daily"), (config.WEEKLY_PERIOD, "weekly")]:
    print(f"{label:<7} (period {period:>3}): {stationarity.seasonal_strength(y_train, period):.3f}")

daily   (period  24): 0.318
weekly  (period 168): 0.393


### 3.3 Stationarity

ADF and KPSS test complementary nulls, so agreement between them is what
justifies the differencing orders. Run on the **training sample only**.

In [23]:
stationarity.stationarity_report(y_train).round(4)

,series,test,null,statistic,p_value,crit_5pct,lags_used,reject_null_5pct
0,level,ADF,unit root present,-8.7610,0.0,-2.8625,28,True
1,level,KPSS,series is stationary,0.0614,0.1,0.4630,23,False
2,first_difference,ADF,unit root present,-15.9483,0.0,-2.8625,28,True
3,first_difference,KPSS,series is stationary,0.0417,0.1,0.4630,68,False
4,seasonal_difference_24,ADF,unit root present,-12.6725,0.0,-2.8625,25,True
5,seasonal_difference_24,KPSS,series is stationary,0.0150,0.1,0.4630,24,False
6,both_differences_24,ADF,unit root present,-17.4448,0.0,-2.8625,28,True
7,both_differences_24,KPSS,series is stationary,0.0611,0.1,0.4630,71,False


## 4. Benchmark models

Five rules under the rolling-origin protocol: 14 origins spaced 24 hours apart,
each issuing a 24-step forecast.

In [24]:
forecasts = {}

for name, fn in benchmarks.benchmark_suite(config.DAILY_PERIOD, config.WEEKLY_PERIOD).items():
    forecasts[name] = benchmarks.rolling_origin_forecast(y, test_index, config.HORIZON, fn)

evaluation.evaluate_all(forecasts, y_test, y_train).round(3)

,model,MAE,RMSE,MASE,Bias
0,seasonal_naive_weekly,43.457,81.409,0.813,-13.160
1,seasonal_naive_daily,48.309,85.565,0.904,1.751
2,mean,50.258,74.938,0.941,-3.287
3,naive,85.551,110.390,1.601,50.977
4,drift,85.802,110.679,1.606,51.368


**Protocol check.** Perturbing the first block's actuals must not change the
first block's forecast, but must change later blocks.

In [25]:
tampered = y.copy()
tampered.loc[test_index[:24]] += 5000

base = benchmarks.rolling_origin_forecast(y, test_index, 24, benchmarks.naive_forecast)
alt  = benchmarks.rolling_origin_forecast(tampered, test_index, 24, benchmarks.naive_forecast)

print("block 1 unchanged:", np.allclose(base.iloc[:24], alt.iloc[:24]))
print("block 2 changed:  ", not np.allclose(base.iloc[24:48], alt.iloc[24:48]))

block 1 unchanged: True
block 2 changed:   True


## 5. SARIMAX

Weather enters as exogenous regressors. The model is run twice — once with
realised test-set weather (conditional) and once persisting the last pre-origin
observation (operational) — to quantify what a missing weather forecast costs.

In [26]:
weather = [c for c in config.WEATHER_COLS if c in frame.columns]
exog = frame[weather]

fit = sarimax.fit_sarimax(y_train, exog_train=exog.loc[y_train.index])
print(f"AIC {fit.aic:.1f}   BIC {fit.bic:.1f}")

/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency h will be used.
  self._init_dates(dates, freq)
/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency h will be used.
  self._init_dates(dates, freq)


  [SARIMAX] DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to re
AIC 32506.5   BIC 32572.3


Coefficient estimates. Note the standard errors on the weather terms relative
to the coefficients — the model cannot identify them, which anticipates the
result below.

In [27]:
fit.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                                     SARIMAX Results                                      
==========================================================================================
Dep. Variable:                         Appliances   No. Observations:                 2954
Model:             SARIMAX(1, 0, 1)x(1, 0, 1, 24)   Log Likelihood              -16242.242
Date:                            Thu, 13 Aug 2026   AIC                          32506.484
Time:                                    09:29:19   BIC                          32572.287
Sample:                                01-11-2016   HQIC                         32530.182
                                     - 05-13-2016                                         
Covariance Type:                              opg                                         
==============================================================================
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
intercept      0.3891      0.322      1.209      0.227      -0.242       1.020
T_out         -2.1702      4.711     -0.461      0.645     -11.404       7.063
RH_out        -0.4621      0.994     -0.465      0.642      -2.409       1.485
Windspeed      1.4879      1.136      1.310      0.190      -0.738       3.714
Visibility    -0.0402      0.141     -0.285      0.776      -0.317       0.236
Tdewpoint      1.8239      4.881      0.374      0.709      -7.743      11.391
ar.L1          0.7749      0.016     48.436      0.000       0.744       0.806
ma.L1         -0.3889      0.022    -17.339      0.000      -0.433      -0.345
ar.S.L24       0.9873      0.004    280.752      0.000       0.980       0.994
ma.S.L24      -0.9412      0.008   -121.828      0.000      -0.956      -0.926
sigma2      3783.0995     57.575     65.708      0.000    3670.256    3895.944
===================================================================================
Ljung-Box (L1) (Q):                   1.74   Jarque-Bera (JB):             11182.93
Prob(Q):                              0.19   Prob(JB):                         0.00
Heteroskedasticity (H):               0.70   Skew:                             2.08
Prob(H) (two-sided):                  0.00   Kurtosis:                        11.62
===================================================================================

Warnings:
[1] Covariance matrix calculated using the outer product of gradients (complex-step).
"""

In [28]:
forecasts["sarimax_conditional"] = sarimax.rolling_origin_sarimax(
    fit, y, test_index, config.HORIZON, exog=exog)

forecasts["sarimax_operational"] = sarimax.rolling_origin_sarimax(
    fit, y, test_index, config.HORIZON,
    exog=sarimax.persisted_exog(exog, test_index, config.HORIZON))

evaluation.evaluate_all(
    {k: v for k, v in forecasts.items() if "sarimax" in k}, y_test, y_train
).round(3)

,model,MAE,RMSE,MASE,Bias
0,sarimax_conditional,37.849,66.407,0.708,-5.139
1,sarimax_operational,38.429,66.347,0.719,-3.927


## 6. Feature-based model

Direct multi-horizon gradient boosting. Every target-derived feature is anchored
to the forecast **origin** rather than the target timestamp, so leakage is
structurally impossible at any lead time.

In [29]:
indoor  = [c for c in config.INDOOR_COLS if c in frame.columns]

design, feature_cols = feature_models.build_design(
    y, exog_origin=frame[indoor + weather], exog_future=None,
    max_horizon=config.HORIZON)

print(f"{design.shape[0]} rows x {len(feature_cols)} features")

74652 rows x 47 features


**Leakage check.** Perturb every test observation; no training-row feature may
respond.

In [30]:
tampered = y.copy()
tampered.loc[test_index] += 10_000

alt_design, _ = feature_models.build_design(
    tampered, exog_origin=frame[indoor + weather], exog_future=None,
    max_horizon=config.HORIZON)

mask = design.index < test_index[0]
cols = [c for c in feature_cols if c != "horizon"]
print("training features unchanged:",
      np.allclose(design.loc[mask, cols], alt_design.loc[mask, cols]))

training features unchanged: True


**Iteration count** is selected on a chronological holdout. scikit-learn's
built-in early stopping is disabled: it splits validation rows at random, and
since each timestamp contributes 24 near-identical rows, that split leaks.

In [31]:
train_rows = design.loc[design.index < test_index[0]]
X_train, y_train_ml = train_rows[feature_cols], train_rows["target"]

chosen = feature_models.select_n_iter(X_train, y_train_ml)
print(f"selected iterations : {chosen['n_iter']}")
print(f"holdout MAE         : {chosen['holdout_mae']:.2f}")

curve = chosen["curve"]
print("\nholdout MAE by iteration:")
for i in (1, 25, 50, 100, 200, 400, 800, 1500):
    if i <= len(curve):
        print(f"  {i:>5} : {curve[i-1]:.2f}")

selected iterations : 28
holdout MAE         : 40.55

holdout MAE by iteration:
      1 : 47.26
     25 : 40.57
     50 : 42.56
    100 : 44.14
    200 : 45.47
    400 : 47.06
    800 : 47.71
   1500 : 48.48


In [32]:
model = feature_models.fit_feature_model(X_train, y_train_ml)

forecasts["feature_model"] = feature_models.rolling_origin_predict(
    model, design, feature_cols, test_index, config.HORIZON)

evaluation.evaluate_all({"feature_model": forecasts["feature_model"]}, y_test, y_train).round(3)

,model,MAE,RMSE,MASE,Bias
0,feature_model,37.889,64.987,0.709,-3.998


### Permutation importance

Preferred to split-count importance, since nine indoor temperature sensors in
one dwelling are close to collinear.

In [33]:
test_rows = features.select_rows(
    design, features.rolling_origin_pairs(test_index, config.HORIZON))

importance = feature_models.feature_importance(
    model, test_rows[feature_cols], test_rows["target"])

importance.head(10).round(3)

,feature,importance,std
0,hour,12.488,0.579
1,hour_sin,2.444,0.580
2,dow_sin,2.139,0.473
3,RH_5_origin,0.638,0.224
4,roll_std_168,0.557,0.182
5,dayofweek,0.381,0.463
6,RH_3_origin,0.271,0.205
7,hour_cos,0.242,0.370
8,T7_origin,0.094,0.085
9,roll_std_24,0.088,0.057


In [34]:
calendar = {"hour","hour_sin","hour_cos","dow_sin","dow_cos","dayofweek","is_weekend","horizon"}
positive = importance["importance"].clip(lower=0).sum()

print(f"calendar features     {importance[importance.feature.isin(calendar)]['importance'].clip(lower=0).sum()/positive:.1%}")
print(f"target lags / rolling {importance[importance.feature.str.startswith(('lag_','roll_'))]['importance'].clip(lower=0).sum()/positive:.1%}")
print(f"lagged exogenous      {importance[importance.feature.str.endswith('_origin')]['importance'].clip(lower=0).sum()/positive:.1%}")

calendar features     90.8%
target lags / rolling 3.3%
lagged exogenous      5.9%


In [35]:
fig = plotting.plot_feature_importance(importance)

## 7. Foundation model

Chronos-T5 applied zero-shot. **Univariate** — it sees only the target history,
no calendar features, no weather, no sensors. Given that calendar features carry
roughly 90% of the importance above, that is a severe handicap.

In [36]:
import time

t0 = time.time()
chronos = foundation.chronos_forecaster(device="auto")
forecasts["chronos_zeroshot"] = benchmarks.rolling_origin_forecast(
    y, test_index, config.HORIZON, chronos)
print(f"{time.time() - t0:.0f}s for {config.N_ORIGINS} origins")

evaluation.evaluate_all({"chronos_zeroshot": forecasts["chronos_zeroshot"]}, y_test, y_train).round(3)

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

62s for 14 origins


,model,MAE,RMSE,MASE,Bias
0,chronos_zeroshot,36.16,74.166,0.677,-28.354


### Sampling variance

Chronos samples stochastically, so its point forecast carries Monte Carlo error.
If the spread across seeds is comparable to the gap between models, the ranking
is partly noise.

In [37]:
runs = []
for seed in range(3):
    np.random.seed(seed)
    fn = foundation.chronos_forecaster(device="auto")
    pred = benchmarks.rolling_origin_forecast(y, test_index, config.HORIZON, fn)
    m = evaluation.mase(y_test, pred, y_train)
    runs.append(m)
    print(f"seed {seed}: MASE {m:.4f}")

print(f"\nspread across seeds: {max(runs) - min(runs):.4f}")

seed 0: MASE 0.6741
seed 1: MASE 0.6754
seed 2: MASE 0.6821

spread across seeds: 0.0081


### Interval calibration

Chronos is the only model here producing a predictive distribution natively.
Nominal coverage is 80%.

In [38]:
quantiles = pd.concat(fn.quantiles_, ignore_index=True)
quantiles.index = test_index

coverage = ((y_test >= quantiles["q0.1"]) & (y_test <= quantiles["q0.9"])).mean()
print(f"empirical coverage of the nominal 80% interval: {coverage:.1%}")

fig, ax = plt.subplots(figsize=(13, 5))
ax.plot(test_index[:72], y_test.iloc[:72], color="black", lw=2, label="actual")
ax.plot(test_index[:72], pred.iloc[:72], color="tab:red", lw=1.3, label="median")
ax.fill_between(test_index[:72], quantiles["q0.1"].iloc[:72],
                quantiles["q0.9"].iloc[:72], alpha=0.2, color="tab:red", label="10-90%")
ax.legend(frameon=False)
ax.set_title("Chronos zero-shot, first 72 test hours")
plt.show()

empirical coverage of the nominal 80% interval: 59.8%


## 8. Comparison and significance testing

### 8.1 All models

In [39]:
results = evaluation.evaluate_all(forecasts, y_test, y_train)
results.round(3)

,model,MAE,RMSE,MASE,Bias
0,chronos_zeroshot,36.160,74.166,0.677,-28.354
1,sarimax_conditional,37.849,66.407,0.708,-5.139
2,feature_model,37.889,64.987,0.709,-3.998
3,sarimax_operational,38.429,66.347,0.719,-3.927
4,seasonal_naive_weekly,43.457,81.409,0.813,-13.160
5,seasonal_naive_daily,48.309,85.565,0.904,1.751
6,mean,50.258,74.938,0.941,-3.287
7,naive,85.551,110.390,1.601,50.977
8,drift,85.802,110.679,1.606,51.368


### 8.2 Diebold–Mariano tests

Point estimates alone will mislead here. Loss differentials are serially
correlated within each 24-hour block, so the variance needs a Newey–West
estimator.

In [40]:
from scipy import stats

def diebold_mariano(actual, f1, f2, h=24):
    d = ((actual - f1).abs() - (actual - f2).abs()).dropna()
    n = len(d)
    var = d.var(ddof=0)
    for lag in range(1, h):
        var += 2 * (1 - lag / h) * np.cov(d[lag:], d[:-lag])[0, 1]
    dm = d.mean() / np.sqrt(var / n)
    return dm, 2 * (1 - stats.norm.cdf(abs(dm)))

best = results["model"].iloc[0]
print(f"Leading model: {best}\n")

for other in results["model"].iloc[1:]:
    dm, p = diebold_mariano(y_test, forecasts[best], forecasts[other])
    verdict = "SIGNIFICANT" if p < 0.05 else "not significant"
    print(f"  vs {other:<24} DM {dm:+6.2f}  p {p:.4f}  {verdict}")

Leading model: chronos_zeroshot

  vs sarimax_conditional      DM  -0.76  p 0.4478  not significant
  vs feature_model            DM  -0.51  p 0.6088  not significant
  vs sarimax_operational      DM  -1.06  p 0.2898  not significant
  vs seasonal_naive_weekly    DM  -1.24  p 0.2145  not significant
  vs seasonal_naive_daily     DM  -2.48  p 0.0131  SIGNIFICANT
  vs mean                     DM  -6.94  p 0.0000  SIGNIFICANT
  vs naive                    DM  -4.03  p 0.0001  SIGNIFICANT
  vs drift                    DM  -4.02  p 0.0001  SIGNIFICANT


### 8.3 Forecasts against actuals

In [41]:
forecast_df = pd.DataFrame({"actual": y_test})
for name, pred_series in forecasts.items():
    forecast_df[name] = pred_series.reindex(test_index)

fig = plotting.plot_forecast_window(forecast_df, window_hours=72)

In [42]:
fig = plotting.plot_forecast_panel(forecast_df)

### 8.4 Error growth with lead time

Note the confound: origins fall at a fixed hour and are spaced at exactly 24
hours, so lead time tracks time of day as well as distance from the origin.

In [43]:
lead = evaluation.errors_by_horizon(
    {k: v for k, v in forecasts.items()}, y_test, config.HORIZON)

fig = plotting.plot_error_by_lead_time(lead)
lead.loc[[1, 6, 12, 18, 24]].round(1)

,mean,naive,seasonal_naive_daily,seasonal_naive_weekly,drift,sarimax_conditional,sarimax_operational,feature_model,chronos_zeroshot
lead_time,,,,,,,,,
1,32.1,32.5,37.6,40.1,32.5,16.1,16.0,31.9,26.5
6,42.9,97.1,4.6,4.3,97.3,11.6,12.1,8.4,3.9
12,36.6,90.8,12.9,14.5,91.2,13.1,15.4,6.7,5.5
18,54.2,79.5,81.9,73.0,79.8,55.4,54.5,56.7,55.7
24,62.8,64.4,64.4,83.2,64.5,49.5,49.7,56.6,57.9


### 8.5 Residual diagnostics

In [44]:
from statsmodels.stats.diagnostic import acorr_ljungbox
from statsmodels.tsa.stattools import acf

residuals = forecast_df[best] - forecast_df["actual"]

lb = acorr_ljungbox(residuals, lags=[24, 48], return_df=True)
a = acf(residuals, nlags=48)

print(f"Ljung-Box 24 lags : Q = {lb.lb_stat.iloc[0]:.1f}")
print(f"residual ACF lag 1: {a[1]:.3f}  (95% band +/- {1.96/np.sqrt(len(residuals)):.3f})")
print(f"skewness          : {residuals.skew():.2f}")
print(f"corr(|resid|, act): {residuals.abs().corr(forecast_df['actual']):.3f}")

fig = plotting.plot_residual_diagnostics(residuals)

Ljung-Box 24 lags : Q = 243.8
residual ACF lag 1: 0.563  (95% band +/- 0.107)
skewness          : -2.79
corr(|resid|, act): 0.956


## 9. Save results

Colab wipes its filesystem when the runtime disconnects. Run this before closing
the tab.

In [45]:
forecast_df.to_csv(config.FORECAST_DIR / "all_forecasts.csv")
results.to_csv(config.METRICS_DIR / "model_comparison.csv", index=False)
importance.to_csv(config.METRICS_DIR / "feature_importance.csv", index=False)
print("saved to outputs/")

saved to outputs/


In [46]:
!zip -qr /content/outputs.zip outputs reports
from google.colab import files
files.download("/content/outputs.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>